<a href="https://colab.research.google.com/github/sksreejareddy-sketch/doomscrool-detector/blob/main/doomscroll_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📵 Doom Scroll Detector
Your webcam watches while you study. The moment a phone shows up in frame, it blasts a siren and roasts you.

**How it works in Colab**
- Colab runs in the cloud, so it has no webcam or speakers of its own. The **browser** captures the webcam and plays the sound.
- Each frame is sent to Python, where **YOLO (Ultralytics)** and **OpenCV** look for a `cell phone` (COCO class 67).
- Python replies with the boxes, and the browser draws them, plays the siren and speaks the callout.
- PyGame needs real audio hardware, which Colab doesn't have. This notebook plays the sound through the browser instead (a local PyGame version is easy to add later).

**Before you run:** Runtime → Change runtime type → **T4 GPU** (works on CPU too, just slower). Then run every cell in order and allow camera access when asked.

## 1. Install and imports

In [ ]:
!pip install -q ultralytics
import cv2, numpy as np, base64, io, wave
import pygame  # preinstalled in Colab; used here only to confirm the stack is available
from ultralytics import YOLO
print("OpenCV", cv2.__version__, "| PyGame", pygame.version.ver)

## 2. Load the YOLO model
`yolo11n` is the smallest and fastest model. Swap to `yolo11s.pt` for better accuracy.

In [ ]:
model = YOLO("yolo11n.pt")
PHONE_CLASS = 67   # 'cell phone' in the COCO dataset
CONF = 0.35        # lower = more sensitive (more false alarms)

# warm-up so the first real frame isn't slow
model.predict(np.zeros((480, 640, 3), np.uint8), classes=[PHONE_CLASS], verbose=False)
print("Model ready")

## 3. Generate the siren
A loud two-tone siren built with NumPy and saved as an in-memory WAV. **Turn your volume down before previewing.**

In [ ]:
from IPython.display import Audio

def make_siren(seconds=2.5, sr=22050):
    t = np.linspace(0, seconds, int(sr * seconds), endpoint=False)
    # alternate between two pitches 3 times per second
    freq = np.where(np.sin(2 * np.pi * 3 * t) > 0, 1200, 700)
    phase = 2 * np.pi * np.cumsum(freq) / sr
    wave_ = 0.6 * np.sign(np.sin(phase)) + 0.4 * np.sin(phase)
    pcm = (np.clip(wave_, -1, 1) * 32767 * 0.9).astype(np.int16)
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(pcm.tobytes())
    return buf.getvalue()

siren_bytes = make_siren()
SIREN_B64 = base64.b64encode(siren_bytes).decode()
Audio(siren_bytes)  # optional preview. It IS loud.

## 4. Frame processor (runs in Python)
The browser calls this function for every webcam frame.

In [ ]:
from IPython.display import JSON
from google.colab import output

def process_frame(data_url):
    raw = base64.b64decode(data_url.split(",")[1])
    img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    res = model.predict(img, classes=[PHONE_CLASS], conf=CONF, imgsz=480, verbose=False)[0]
    boxes = []
    for b in res.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        boxes.append({"x1": x1 / w, "y1": y1 / h, "x2": x2 / w, "y2": y2 / h,
                      "conf": float(b.conf[0])})
    return JSON({"phone": len(boxes) > 0, "boxes": boxes})

output.register_callback("notebook.process_frame", process_frame)
print("Callback registered")

## 5. Launch the detector
Click **Start studying**, allow the camera, then pick up your phone. 😈

Tune these inside the cell if needed:
- `CONFIRM_FRAMES`: consecutive frames with a phone before the alarm fires (stops one-frame false positives)
- `COOLDOWN_MS`: minimum gap between alarms while you keep holding the phone

In [ ]:
from IPython.display import display, HTML

PAGE = """
<div style="font-family:system-ui,sans-serif;max-width:660px">
  <div style="margin-bottom:8px">
    <button id="startBtn" style="padding:8px 16px;font-size:15px">▶ Start studying</button>
    <button id="stopBtn" style="padding:8px 16px;font-size:15px" disabled>■ Stop</button>
  </div>
  <div id="stats" style="margin-bottom:8px;font-size:15px">Status: idle</div>
  <div style="position:relative;width:640px;height:480px;background:#111;border-radius:8px;overflow:hidden">
    <video id="v" autoplay playsinline muted width="640" height="480"></video>
    <canvas id="o" width="640" height="480" style="position:absolute;left:0;top:0"></canvas>
    <div id="banner" style="display:none;position:absolute;inset:0;background:rgba(220,0,0,.55);
         color:#fff;font-size:44px;font-weight:800;text-align:center;padding-top:170px;
         text-shadow:2px 2px 6px #000">📵 PUT THE PHONE DOWN!</div>
  </div>
</div>
<script>
(function () {
  const CONFIRM_FRAMES = 2;
  const COOLDOWN_MS = 6000;
  const CLEAR_FRAMES = 4;
  const ROASTS = [
    "Bro. Really? The exam is not going to pass itself.",
    "I saw that. Put the phone down and open the book.",
    "Doom scrolling again? Your future self is disappointed.",
    "That reel will still be there in two hours. Study now.",
    "Caught you. Phone down. Eyes on the notes.",
    "Nothing on that screen is more important than your syllabus."
  ];
  const siren = new Audio("data:audio/wav;base64,__SIREN__");
  const $ = id => document.getElementById(id);
  const v = $("v"), o = $("o"), banner = $("banner"), stats = $("stats");
  const octx = o.getContext("2d");
  let running = false, stream = null;
  let strikes = 0, clear = 0, holding = false, lastAlarm = 0;
  let caught = 0, startTime = 0, frames = 0, lastFpsT = 0, fps = 0;

  function speak(text) {
    try {
      speechSynthesis.cancel();
      const u = new SpeechSynthesisUtterance(text);
      u.rate = 1.05; u.volume = 1;
      speechSynthesis.speak(u);
    } catch (e) {}
  }

  function alarm() {
    const now = Date.now();
    if (now - lastAlarm < COOLDOWN_MS) return;
    lastAlarm = now;
    banner.style.display = "block";
    siren.currentTime = 0;
    siren.play().catch(() => {});
    siren.onended = () => speak(ROASTS[Math.floor(Math.random() * ROASTS.length)]);
  }

  function updateStats() {
    const mins = ((Date.now() - startTime) / 60000).toFixed(1);
    stats.textContent = "Studying: " + mins + " min | Phone pickups: " + caught + " | " + fps + " fps";
  }

  function draw(d) {
    octx.clearRect(0, 0, 640, 480);
    octx.lineWidth = 4; octx.strokeStyle = "#ff2d2d"; octx.fillStyle = "#ff2d2d";
    octx.font = "bold 18px sans-serif";
    d.boxes.forEach(b => {
      const x = b.x1 * 640, y = b.y1 * 480, w = (b.x2 - b.x1) * 640, h = (b.y2 - b.y1) * 480;
      octx.strokeRect(x, y, w, h);
      octx.fillText("PHONE " + Math.round(b.conf * 100) + "%", x, Math.max(18, y - 6));
    });
  }

  async function loop() {
    const cap = document.createElement("canvas");
    cap.width = 640; cap.height = 480;
    const cctx = cap.getContext("2d");
    while (running) {
      cctx.drawImage(v, 0, 0, 640, 480);
      const url = cap.toDataURL("image/jpeg", 0.7);
      let d;
      try {
        const res = await google.colab.kernel.invokeFunction("notebook.process_frame", [url], {});
        d = res.data["application/json"];
      } catch (e) {
        stats.textContent = "Lost connection to the Colab runtime. Re-run this cell.";
        break;
      }
      draw(d);

      if (d.phone) { strikes++; clear = 0; } else { clear++; strikes = 0; }
      if (strikes >= CONFIRM_FRAMES) {
        if (!holding) { holding = true; caught++; }
        alarm();
      }
      if (clear >= CLEAR_FRAMES) {
        holding = false;
        banner.style.display = "none";
      }

      frames++;
      const t = Date.now();
      if (t - lastFpsT > 1000) { fps = frames; frames = 0; lastFpsT = t; }
      updateStats();
    }
    stop();
  }

  function stop() {
    running = false;
    if (stream) { stream.getTracks().forEach(t => t.stop()); stream = null; }
    banner.style.display = "none";
    $("startBtn").disabled = false; $("stopBtn").disabled = true;
  }

  $("startBtn").onclick = async () => {
    try {
      stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 } });
    } catch (e) {
      stats.textContent = "Camera blocked. Allow camera access in the browser and try again.";
      return;
    }
    v.srcObject = stream; await v.play();
    // unlock audio + speech inside this click (browsers require a user gesture)
    siren.volume = 0;
    try { await siren.play(); } catch (e) {}
    siren.pause(); siren.currentTime = 0; siren.volume = 1;
    try { speechSynthesis.speak(new SpeechSynthesisUtterance("")); } catch (e) {}

    strikes = clear = caught = 0; holding = false; lastAlarm = 0;
    startTime = Date.now(); lastFpsT = Date.now();
    $("startBtn").disabled = true; $("stopBtn").disabled = false;
    running = true; loop();
  };
  $("stopBtn").onclick = stop;
})();
</script>
""".replace("__SIREN__", SIREN_B64)

display(HTML(PAGE))

## Troubleshooting
- **Camera won't start:** click the camera icon in the address bar and allow access. Colab must be open in its own tab, not embedded in another page.
- **No sound:** the browser only allows audio after a click, so always use the Start button. Also check the tab isn't muted.
- **Phone not detected:** hold it so the screen or body is clearly visible, then lower `CONF` to `0.25`, or switch to `yolo11s.pt`.
- **Too many false alarms:** raise `CONF` to `0.5`, or raise `CONFIRM_FRAMES` to `3`.
- **Slow / low fps:** use a GPU runtime, or lower `imgsz` in `process_frame` to `320`.
- **Runtime disconnects:** Colab stops idle sessions. Re-run the last cell to restart the detector.

## Ideas to extend it
- Also detect `person` and warn if you leave your seat
- Log pickups to a CSV and chart your focus time per session
- Fine-tune YOLO on photos of *you* holding a phone for better accuracy
- Build a local PyGame + OpenCV version for real-time use outside Colab